# 6.14 · LDA 作为降维 / LDA as Dimensionality Reduction

> **课程定位 / Where this fits**
> 5.12 把 LDA 当**分类器**讲。但 LDA 的内核其实是**有监督降维**: 它找的不是方差最大方向(那是 PCA), 而是"**最能把类别分开**"的方向。本课把这一面单独讲透, 与 PCA(6.8 无监督)正好对照, 收束 Part 6 的降维主线: **PCA 保方差, LDA 保可分性**。
> LDA, beyond classification (5.12), is fundamentally supervised dimensionality reduction: it finds directions that best separate classes. The supervised counterpart to PCA.

> 💡 **面试相关 / Interview-relevant**
> - "LDA 与 PCA 的区别(有监督/无监督, 可分性/方差)" ★★★★★
> - "LDA 的目标函数(类间/类内散度比)" ★★★★★
> - "LDA 最多能降到几维(K-1)及为什么" ★★★★★
> - "LDA 何时优于 PCA 做分类前降维" ★★★★

---

## 学习目标 / Learning Objectives
1. LDA 的 Fisher 判据: 最大化类间/类内散度比。
2. 从零(广义特征问题)实现 LDA 投影。
3. **K-1 维上限**的由来。
4. PCA vs LDA 在分类前降维的对比。

## 目录 / TOC
1. [Fisher 判据 + K-1 上限 ⭐](#1)
2. [🍷 数据: Wine + 从零 ⭐](#2)
3. [LDA vs PCA 投影 ⭐](#3)
4. [降维后分类: LDA vs PCA ⭐](#4)
5. [小结](#5)


<a id="1"></a>
## 1. Fisher 判据 + K-1 上限 ⭐ / Fisher Criterion & K-1 Limit

LDA 找投影方向 $\mathbf{w}$, 让投影后**类间分得开、类内聚得紧**。定义两个散度矩阵:
- **类内散度** $\mathbf{S}_W = \sum_k\sum_{i\in C_k}(\mathbf{x}_i-\boldsymbol\mu_k)(\mathbf{x}_i-\boldsymbol\mu_k)^\top$ (各类内部的离散)。
- **类间散度** $\mathbf{S}_B = \sum_k n_k(\boldsymbol\mu_k-\boldsymbol\mu)(\boldsymbol\mu_k-\boldsymbol\mu)^\top$ (各类均值离总均值多远)。

**Fisher 判据**: 最大化
$$J(\mathbf{w}) = \frac{\mathbf{w}^\top\mathbf{S}_B\mathbf{w}}{\mathbf{w}^\top\mathbf{S}_W\mathbf{w}}$$
解是**广义特征问题** $\mathbf{S}_B\mathbf{w}=\lambda\mathbf{S}_W\mathbf{w}$ 的最大特征向量。对比 PCA 解的是 $\mathbf{C}\mathbf{w}=\lambda\mathbf{w}$(只看总方差, 不看标签)。

**K-1 维上限**(面试常考): $\mathbf{S}_B$ 由 K 个类均值相对总均值构成, 而它们和为 0(线性相关), 故 $\text{rank}(\mathbf{S}_B)\le K-1$ → 最多 **K-1 个非零广义特征值** → LDA 最多降到 **K-1 维**。3 类只能降到 2 维, 2 类只能降到 1 维。


<a id="2"></a>
## 2. 数据: Wine + 从零 ⭐ / Wine & From Scratch

复用 **Wine**(5.12 用过): 13 维, 3 类。LDA 最多降到 3−1=2 维。从零解广义特征问题, 对照 sklearn。


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from scipy.linalg import eigh
sns.set_theme(style="whitegrid")
np.set_printoptions(precision=3, suppress=True)

wine = load_wine(); X = StandardScaler().fit_transform(wine.data); y = wine.target
K = len(np.unique(y)); d = X.shape[1]
print(f"Wine: {X.shape}, {K} 类 → LDA 最多降到 {K-1} 维")

# 从零: 类内/类间散度 + 广义特征问题
mean_all = X.mean(0)
Sw = np.zeros((d,d)); Sb = np.zeros((d,d))
for k in range(K):
    Xk = X[y==k]; muk = Xk.mean(0)
    Sw += (Xk-muk).T @ (Xk-muk)
    nk = len(Xk); diff = (muk-mean_all).reshape(-1,1)
    Sb += nk * diff @ diff.T
# 解 Sb w = λ Sw w, 取最大的 K-1 个
vals, vecs = eigh(Sb, Sw)                 # 升序
W = vecs[:, ::-1][:, :K-1]                # 取最大 K-1 个方向
Z_scratch = X @ W

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
Z_sk = LinearDiscriminantAnalysis(n_components=2).fit_transform(X, y)
print(f"从零 LDA 投影形状 {Z_scratch.shape}; 与 sklearn 各维|相关|:",
      [f"{abs(np.corrcoef(Z_scratch[:,i], Z_sk[:,i])[0,1]):.3f}" for i in range(2)])
print("(≈1 表示从零与 sklearn 求得同样的判别方向, 符号可能相反)")


<a id="3"></a>
## 3. LDA vs PCA 投影 ⭐ / LDA vs PCA Projections


In [ ]:
from sklearn.decomposition import PCA
Z_pca = PCA(n_components=2).fit_transform(X)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
for k,name in enumerate(wine.target_names):
    axes[0].scatter(Z_scratch[y==k,0], Z_scratch[y==k,1], label=name, s=22)
    axes[1].scatter(Z_pca[y==k,0], Z_pca[y==k,1], label=name, s=22)
axes[0].set_title("LDA(有监督): 用标签 → 类别分得最开"); axes[0].legend()
axes[1].set_title("PCA(无监督): 只保方差 → 类别更重叠"); axes[1].legend()
for a in axes: a.set_xlabel("分量1"); a.set_ylabel("分量2")
plt.tight_layout(); plt.show()
print("LDA 用了标签, 投影后三类几乎线性可分; PCA 不看标签, 类别更易重叠")


<a id="4"></a>
## 4. 降维后分类: LDA vs PCA ⭐ / Classification After DR

把降维当分类的**预处理**: 在 2 维上训一个简单分类器, 看 LDA 投影是否比 PCA 投影更利于分类。


In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler as SS

print("在 2 维投影上做 KNN 分类(5-fold CV 准确率):")
# 注意: 降维要放进 CV 防泄漏(LDA 用了标签, 尤其要在训练折内 fit)
lda_pipe = make_pipeline(SS(), LinearDiscriminantAnalysis(n_components=2), KNeighborsClassifier(5))
pca_pipe = make_pipeline(SS(), PCA(2), KNeighborsClassifier(5))
print(f"  LDA(2D) + KNN: {cross_val_score(lda_pipe, wine.data, y, cv=5).mean():.3f}")
print(f"  PCA(2D) + KNN: {cross_val_score(pca_pipe, wine.data, y, cv=5).mean():.3f}")
print("\nLDA 投影更利于分类(它就是为'可分性'优化的); 但 LDA 受 K-1 维上限 + 高斯等协方差假设(5.12)限制")
print("无标签 → 只能 PCA; 有标签且要分类前降维 → LDA 常更优")


<a id="5"></a>
## 5. 小结 / Summary

```
LDA(降维视角): 最大化 Fisher 判据 J=wᵀS_B w / wᵀS_W w (类间/类内散度比)
解广义特征问题 S_B w=λ S_W w 的最大特征向量; PCA 解 C w=λw(只看总方差)
K-1 维上限: rank(S_B)≤K-1(K个类均值线性相关) → 最多降到 K-1 维
PCA(无监督, 保方差) vs LDA(有监督, 保可分性): 分类前降维 LDA 常更优
局限: K-1 上限 + 高斯/等协方差假设(5.12); 无标签时只能用 PCA
```

### 💡 面试速查
1. **LDA=有监督降维**(找最可分方向); PCA=无监督(找最大方差方向)
2. **Fisher 判据**: 类间散度/类内散度比最大; 解广义特征问题
3. **最多降到 K-1 维**(S_B 秩 ≤ K-1)——高频考点
4. **分类前降维 LDA 常优于 PCA**(直接优化可分性)
5. 受高斯+等协方差假设约束; 无标签退回 PCA

### 下一节
**6.15 自编码器**——用神经网络做非线性降维: encoder 压到低维瓶颈, decoder 重构, 是 PCA 的非线性深度学习版, 也是生成模型(VAE)的前身。
